In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("180").setMaster("local[4]")
spark = SparkSession.builder.config(conf = conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/14 01:09:14 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface enp0s3)
25/08/14 01:09:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/14 01:09:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: Logs

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| num         | varchar |
+-------------+---------+
In SQL, id is the primary key for this table.
id is an autoincrement column.
 

Find all numbers that appear at least three times consecutively.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Logs table:
+----+-----+
| id | num |
+----+-----+
| 1  | 1   |
| 2  | 1   |
| 3  | 1   |
| 4  | 2   |
| 5  | 1   |
| 6  | 2   |
| 7  | 2   |
+----+-----+
Output: 
+-----------------+
| ConsecutiveNums |
+-----------------+
| 1               |
+-----------------+
Explanation: 1 is the only number that appears consecutively for at least three times.
'''

In [2]:
data = [
(1,1),
(2,1),
(3,1),
(4,2),
(5,1),
(6,2),
(7,2)  
]
schema = ['id','num']

In [3]:
df = spark.createDataFrame(data = data, schema = schema)
df.show()

+---+---+
| id|num|
+---+---+
|  1|  1|
|  2|  1|
|  3|  1|
|  4|  2|
|  5|  1|
|  6|  2|
|  7|  2|
+---+---+



In [4]:
from pyspark.sql.window import Window

window_spec = Window.orderBy(F.col("id"))

In [7]:
temp_df = df.select(F.col("id"),F.col("num"),
                    F.lag(F.col("num"),1,None).over(window_spec).alias("LAG1"),
                    F.lag(F.col("num"),2,None).over(window_spec).alias("LAG2"),
                   )
temp_df.where((F.col("num") == F.col("LAG1")) & (F.col("num") == F.col("LAG2")))\
       .select(F.col("num").alias("ConsecutiveNums"))\
       .show()

25/08/14 01:26:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/14 01:26:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/14 01:26:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/14 01:26:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/14 01:26:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------+
|ConsecutiveNums|
+---------------+
|              1|
+---------------+



## SQL Solution
<pre>
WITH TEMP AS (
    SELECT id, num, 
           LAG(num,1,NULL) OVER(ORDER BY id) as LAG1,
           LAG(num,2,NULL) OVER(ORDER BY id) as LAG2
    FROM medium_180
)
SELECT num as ConsecutiveNums 
FROM TEMP 
WHERE num = LAG1 and num = LAG2
</pre>